In [ ]:
# basically make a rolling averages of the baseline and the data and see if there is a spike in the current data.
#use the slopes of the rolling averages to see if brooding is starting or if its just a random spike
#also detect if the temperature is stable or not


In [ ]:


import pandas as pd
import numpy as np
import os
import glob


ROLLING_WINDOW  = 5     # readings to smooth (~1 hour)
SLOPE_LOOKBACK  = 48    # readings back for slope calc (12 hours)
SUSTAINED       = 96    # consecutive readings required to confirm brooding (24 hours)
TREND_READINGS  = 48    # consecutive rising readings for pre-brooding warning (12 hours)

# --- Brooding target window ---
BROODING_TEMP_TARGET    = 93.5   # °F midpoint of brood nest range

#temp variability for ambient temp instead of the direct temp
# temp average

BROODING_TEMP_TOLERANCE = 5.0    # °F — so 88.5 to 98.5°F counts
BROODING_TEMP_LOW  = BROODING_TEMP_TARGET - BROODING_TEMP_TOLERANCE   # 88.5°F
BROODING_TEMP_HIGH = BROODING_TEMP_TARGET + BROODING_TEMP_TOLERANCE   # 98.5°F

# Brooding humidity range — bees also regulate humidity in the brood nest
BROODING_HUM_TARGET    = 55.0    # %RH midpoint
BROODING_HUM_TOLERANCE = 5.0     # %RH — so 50-60% RH counts
BROODING_HUM_LOW  = BROODING_HUM_TARGET - BROODING_HUM_TOLERANCE      # 50%
BROODING_HUM_HIGH = BROODING_HUM_TARGET + BROODING_HUM_TOLERANCE      # 60%

# Pre-brooding: temp is warming toward the brooding range but not there yet.
PRE_BROODING_APPROACH_WINDOW = 10.0  # °F below BROODING_TEMP_LOW

# Failing thermoregulation: hive was in brooding range but temp is now erratic or dropping.
FAILING_DROP     = 4.0   # °F drop from recent peak while still elevated

# ==============================================================================
# PROCESS ONE HIVE FILE
# ==============================================================================

def process_hive(filepath):

    df = pd.read_csv(filepath, parse_dates=['Timestamp'])
    df = df.sort_values('Timestamp').reset_index(drop=True)

    hive_name = os.path.splitext(os.path.basename(filepath))[0]
    print(f"  Processing: {hive_name}  ({len(df)} readings, "
          f"{df['Timestamp'].min().date()} → {df['Timestamp'].max().date()})")

    #smooth the temp to detect trends
    df['temp_smooth']     = df['Temperature_Fahrenheit'].rolling(ROLLING_WINDOW, min_periods=1).mean()
    df['humidity_smooth'] = df['Relative_Humidity'].rolling(ROLLING_WINDOW, min_periods=1).mean()

    # is it within the brooding tolerances
    df['temp_in_range'] = (
        (df['temp_smooth'] >= BROODING_TEMP_LOW) &
        (df['temp_smooth'] <= BROODING_TEMP_HIGH)
    )
    df['hum_in_range'] = (
        (df['humidity_smooth'] >= BROODING_HUM_LOW) &
        (df['humidity_smooth'] <= BROODING_HUM_HIGH)
    )
    df['is_in_brooding_range'] = df['temp_in_range'] & df['hum_in_range']

    #is the hive in the brooding range for a while or just a spike
    df['brooding_confirmed'] = (
        df['is_in_brooding_range']
        .rolling(SUSTAINED, min_periods=SUSTAINED)
        .sum() == SUSTAINED
    )

    #is it trending twards the brooding temperature?
    df['slope'] = df['temp_smooth'] - df['temp_smooth'].shift(SLOPE_LOOKBACK)

    approaching = (
        (df['temp_smooth'] >= BROODING_TEMP_LOW - PRE_BROODING_APPROACH_WINDOW) &
        (df['temp_smooth'] <  BROODING_TEMP_LOW)
    )
    consistent_rise = (
        (df['slope'] > 0)
        .rolling(TREND_READINGS, min_periods=TREND_READINGS)
        .sum() == TREND_READINGS
    )
    df['pre_brooding'] = approaching & consistent_rise

    # fi the temperature or humidity are not stable, that means something is up with the hive. 
    df['temp_variance'] = (
        df['Temperature_Fahrenheit']
        .rolling(SUSTAINED, min_periods=SUSTAINED)
        .std()
    )
    df['temp_peak_24h']  = df['temp_smooth'].rolling(96, min_periods=1).max()
    df['drop_from_peak'] = df['temp_peak_24h'] - df['temp_smooth']

    df['is_failing'] = (
        df['brooding_confirmed'] &          # only after confirmed brooding
        (
            (df['temp_variance'] >= FAILING_VARIANCE) |
            (df['drop_from_peak'] >= FAILING_DROP)
        )
    )

    # ------------------------------------------------------------------
    # Step 6: Label each day
    # Priority: failing > brooding > pre_brooding > normal
    # ------------------------------------------------------------------
    df['date'] = df['Timestamp'].dt.date

    daily_rows = []
    for date, group in df.groupby('date'):
        n = len(group)
        label = (
            'failing_thermoregulation' if group['is_failing'].any()         else
            'brooding'                 if group['brooding_confirmed'].any()  else
            'pre_brooding'             if group['pre_brooding'].any()        else
            'normal'
        )
        daily_rows.append({
            'date':             date,
            'label':            label,
            'avg_temp_F':       round(group['temp_smooth'].mean(), 2),
            'avg_humidity':     round(group['humidity_smooth'].mean(), 2),
            'pct_day_brooding': round(group['brooding_confirmed'].sum() / n * 100, 1),
            'pct_day_failing':  round(group['is_failing'].sum() / n * 100, 1),
            'alive':            int(group['Alive_or_Dead'].mode()[0]),
            'total_readings':   n,
        })

    return hive_name, pd.DataFrame(daily_rows)

#repeat the process for every hive in the folder.
def run(input_folder, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    csv_files = glob.glob(os.path.join(input_folder, '*.csv'))

    if not csv_files:
        print(f"No CSV files found in: {input_folder}")
        return

    print("=" * 60)
    print(f"BEEHIVE BROODING DETECTOR")
    print(f"Brooding temp window : {BROODING_TEMP_LOW}°F – {BROODING_TEMP_HIGH}°F")
    print(f"Brooding hum window  : {BROODING_HUM_LOW}% – {BROODING_HUM_HIGH}%")
    print(f"Must sustain for     : {SUSTAINED} readings ({SUSTAINED//4} hours)")
    print(f"Input folder : {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"Hives found  : {len(csv_files)}")
    print("=" * 60)
    print()

    summary_rows = []

    for filepath in sorted(csv_files):
        hive_name, daily_df = process_hive(filepath)

        out_path = os.path.join(output_folder, f"{hive_name}_labels.csv")
        daily_df.to_csv(out_path, index=False)

        counts = daily_df['label'].value_counts()
        for label, count in counts.items():
            print(f"    {label}: {count} days")
        print()

        for _, row in daily_df.iterrows():
            summary_rows.append({'hive': hive_name, **row})

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(output_folder, '_all_hives_summary.csv')
    summary_df.to_csv(summary_path, index=False)

    print(f"All done. Results saved to: {output_folder}")
    print(f"Combined summary: {summary_path}")


#running it. 
if __name__ == '__main__':

    INPUT_FOLDER  = 'data\Beehives_consolodated'
    OUTPUT_FOLDER = 'hive_results'

    run(INPUT_FOLDER, OUTPUT_FOLDER)


<>:232: SyntaxWarning: invalid escape sequence '\B'
<>:232: SyntaxWarning: invalid escape sequence '\B'
C:\Users\vitya\AppData\Local\Temp\ipykernel_68008\1344371418.py:232: SyntaxWarning: invalid escape sequence '\B'
  INPUT_FOLDER  = 'data\Beehives_consolodated'


BEEHIVE BROODING DETECTOR
Brooding temp window : 88.5°F – 98.5°F
Brooding hum window  : 50.0% – 60.0%
Must sustain for     : 96 readings (24 hours)
Input folder : data\Beehives_consolodated
Output folder: hive_results
Hives found  : 17

  Processing: WAC  (4033 readings, 2025-11-08 → 2025-12-20)
    normal: 43 days

  Processing: WAE  (4033 readings, 2025-11-08 → 2025-12-20)
    normal: 43 days

  Processing: WBC  (3644 readings, 2025-11-08 → 2025-12-16)
    normal: 39 days

  Processing: WBE  (3645 readings, 2025-11-08 → 2025-12-16)
    normal: 39 days

  Processing: WCC  (2424 readings, 2025-11-08 → 2025-12-03)
    normal: 26 days

  Processing: WCE  (4032 readings, 2025-11-08 → 2025-12-20)
    normal: 43 days

  Processing: WDC  (4033 readings, 2025-11-08 → 2025-12-20)
    normal: 43 days

  Processing: WDE  (4033 readings, 2025-11-08 → 2025-12-20)
    normal: 43 days

  Processing: WEC  (4033 readings, 2025-11-08 → 2025-12-20)
    normal: 43 days

  Processing: WEE  (3643 readings,

In [2]:
# remove the pre-brooding when there is no brooding after
import os
import pandas as pd
for hive in os.listdir("hive_results"):
    df = pd.read_csv(os.path.join("hive_results", hive))
    for i in range(len(df)-1):
        if df.loc[i, 'label'] == 'pre_brooding' and df.loc[i+7, 'label'] != 'brooding':
            df.loc[i, 'label'] = 'normal'
    df.to_csv(os.path.join("hive_results", hive), index=False)
